# Assistant RAG pour le Code général des impôts marocain

Cette démonstration porte sur les éditions françaises complètes du CGI marocain 2023, 2024, 2025 et 2026. Une question est posée sur **une édition choisie** ; la réponse renvoie à l’article, aux passages cités et aux pages du PDF source.

Le notebook lit les artefacts de la chaîne active. Les cellules ordinaires ne relancent ni l’extraction, ni l’indexation, ni le modèle génératif. La question libre en fin de parcours est optionnelle.

**Parcours :** sources → articles → recherche → réponse citée → évaluation → question libre.


## 1. Architecture retenue

```text
PDF des quatre éditions → articles, notes, hiérarchie et pages
                              ↓
                      passages indexés
                         ├─ BM25
                         └─ E5 dans Chroma
                              ↓
                   fusion RRF + filtre d’édition
                              ↓
                      contexte parent borné
                              ↓
                     Qwen local via Ollama
                              ↓
               réponse et citations en blocs
                              ↓
                    contrôle des preuves
                              ↓
                 réponse / abstention / rejet
```

Streamlit présente le résultat ; MLflow conserve les campagnes et leurs artefacts. La configuration active est désignée dans `data/current_run.json`. Le découpage employé dans les mesures présentées ici est resté constant.


## 2. Préparer la démonstration

Les modules Python sont présents dans `rag_fiscal/`, l’interface dans `streamlit_app.py` et les commandes de lancement dans `scripts/`. Le notebook importe ces fichiers ; il ne les génère pas. Les PDF, les articles, l’index et les résultats mesurés se trouvent dans le même dossier de projet.

Sur ce PC, choisir `.venv/Scripts/python.exe` comme noyau Jupyter. Les cellules de lecture ne relancent ni l’extraction ni l’indexation.


In [ ]:
from pathlib import Path
import hashlib, json, sys

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
candidates = [Path.cwd(), Path.cwd() / "presentation_finale", *Path.cwd().parents]
ROOT = next(p.resolve() for p in candidates if (p / "demo.ipynb").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag_fiscal.candidate_retrieve import CandidateRetriever

print("Dossier du projet :", ROOT)
print("Moteur quatre éditions disponible :", (ROOT / "rag_fiscal/candidate_retrieve.py").is_file())


### Modèle de recherche dense

Le modèle E5 local sert uniquement lors d’une recherche dense ou hybride en direct. Les résultats enregistrés restent consultables sans le charger. Les poids du modèle sont déjà présents dans ce dossier local ; un transfert vers une autre machine exige de préparer séparément les modèles et l’environnement Python.


In [ ]:
protocol = json.loads((ROOT / "data/search_reference/protocol.json").read_text(encoding="utf-8"))
model_dir = ROOT / protocol["model_dir"]
print("Modèle E5 local :", model_dir)
print("Fichiers attendus présents :", all((model_dir / name).is_file() for name in protocol["model_files"]))


## 3. Sources et structure juridique

Les quatre PDF d’origine sont dans `sources/`. Le corpus conserve la hiérarchie des dispositions, les notes séparées, la page du fichier PDF et la pagination imprimée. L’empreinte de chaque PDF est contrôlée contre le manifeste local avant la démonstration.


In [ ]:
def read_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8-sig"))

def read_jsonl(path):
    with (ROOT / path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

state = read_json("data/current_run.json")
assert state["scope"] == "full_cgi_candidate"
manifest = read_json("sources/sources_manifest.json")
for item in manifest:
    source = ROOT / "sources" / item["file"]
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    print(f'CGI {item["year"]} :', "PDF vérifié" if digest.lower() == item["sha256"].lower() else "EMPREINTE DIFFÉRENTE")
articles = read_jsonl(state["active_extraction"] + "/articles.jsonl")
article = next(a for a in articles if a["edition"] == 2026 and a["article"] == "111")
print("Exemple : article", article["article"], "—", article["title"])
print("Hiérarchie :", " > ".join(h["label"] for h in article["hierarchy"]))
print("Pages PDF :", article["pdf_page_start"], "à", article["pdf_page_end"])
print("Pages imprimées :", article["printed_page_start"], "à", article["printed_page_end"])


## 4. Recherche dans la bonne édition

BM25 classe les formulations exactes ; E5 apporte la similarité sémantique. Le moteur fusionne leurs rangs après filtrage par édition et conserve une diversité d’articles dans les cinq premiers passages. Ici, un résultat enregistré est comparé à une recherche BM25 locale. Cette cellule ne charge pas E5 ni Ollama.


In [ ]:
campaign = ROOT / "data/evaluation/test_supplementaire"
questions = {q["id"]: q for q in read_json("benchmark/questions_test_supplementaire.json")["questions"]}
case_id = "ht07"
question = questions[case_id]
record = read_json(str((campaign / "retrieval" / f"{case_id}.json").relative_to(ROOT)))
print("Question :", question["question"])
print("Preuve attendue :", [(e["article"], e["pdf_pages"]) for e in question["evidence"]])
print("Hybride enregistré :", [(p["rank"], p["article"]) for p in record["top5"]])
retriever = CandidateRetriever(backend="chroma")
lexical = retriever.search(question["question"], question["edition"], method="bm25")
print("BM25 recalculé :", [(p["rank"], p["article"]) for p in lexical["results"]])


## 5. Contexte et réponse citée

Les passages retrouvés servent de points d’entrée. Le contexte parent restitue les lignes utiles autour d’eux, notamment l’introduction qui gouverne une liste, dans le budget du modèle. Qwen rédige à partir de ce contexte ; les citations sont ensuite reconstruites et contrôlées à partir des lignes du PDF.

La cellule suivante lit une réponse enregistrée et ne relance pas la génération.


In [ ]:
answer_record = read_json(str((campaign / "answers" / f"{case_id}.json").relative_to(ROOT)))
result = answer_record["result"]
print("Mode de contexte :", result["context"]["mode"])
print("Réponse :", result["answer"])
for citation in result["citations"]:
    print(f'[{citation["id"]}] CGI {citation["edition"]}, article {citation["article"]}, PDF {citation["pdf_page_start"]}–{citation["pdf_page_end"]}, pages imprimées {citation["printed_page_start"]}–{citation["printed_page_end"]}')
    print("   ", " ".join(citation["quote"].split())[:280], "…")


## 6. Une abstention malgré la présence de la preuve

Un modèle utile n’est pas parfait. Le système doit pouvoir s’abstenir lorsque la preuve manque ; il peut aussi s’abstenir à tort. Le cas ci-dessous montre cette seconde limite dans le lot supplémentaire : l’article pertinent était premier et son texte donnait explicitement l’exception relative à la TVA.


In [ ]:
failure = read_json(str((campaign / "answers/ht04.json").relative_to(ROOT)))
print("Question :", failure["question"])
print("Statut :", failure["status"])
print("Message :", failure["result"]["answer"])
print("Article du premier passage :", failure["result"]["retrieval"]["results"][0]["article"])
print("Début du contexte :", failure["result"]["context"]["passages"][0]["text"][:400])


## 7. Évaluation

La campagne principale réunit des questions de développement et d’exploration. Le lot supplémentaire couvre d’autres articles et familles et a été figé avant son exécution. Les mesures séparent la présence d’une preuve parmi les cinq premiers passages, le statut de réponse, la validité structurelle des citations et leur contenu. Les deux lots restent trop limités pour estimer l’exactitude sur l’ensemble du CGI.


In [ ]:
main = read_json("data/evaluation/campagne_principale/final_summary.json")
search = read_json(str((campaign / "retrieval_summary.json").relative_to(ROOT)))
generation = read_json(str((campaign / "generation_summary.json").relative_to(ROOT)))
audit = read_json(str((campaign / "answer_audit.json").relative_to(ROOT)))
print("Campagne principale :", main["answer_status"]["answer"], "/", main["question_count"], "réponses délivrées")
print("Lot supplémentaire :", search["metrics"]["strict_recall_at_5"], "Recall@5 strict sur", search["n"], "questions")
print("Statuts du lot supplémentaire :", generation["counts"])
print("Réponses dont le contenu principal est appuyé :", audit["core_supported_answers"], "/", audit["question_count"])
print("Abstentions injustifiées observées :", audit["false_abstentions"])


**Lecture des résultats.** Le moteur retrouve et cite la preuve dans la plupart des cas de ces lots, mais une question a conduit à une abstention alors que la preuve figurait dans le contexte. Une autre a reçu une réponse appuyée par des lignes valables différentes de la preuve annotée : la mesure stricte du passage attendu doit donc être lue avec son analyse par question. Les deux rejets conservés dans la campagne principale sont des limites acceptées, pas un objectif de perfection à masquer.


## 8. Poser une nouvelle question (optionnel)

Démarrer Ollama, passer `RUN_LIVE` à `True`, puis exécuter cette cellule pour appeler le modèle local sur le GPU. Cette action est désactivée par défaut ; la lecture des résultats enregistrés ne requiert pas le serveur.


In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    from rag_fiscal.ask import answer_question
    config = read_json("config/system.json")
    live = answer_question(
        CandidateRetriever(backend=config["vector_backend"]),
        "Selon le CGI 2026, quel est le délai de déclaration trimestrielle ?",
        2026,
        method="hybrid_rrf", edition_filter="on",
        context_mode=config["context_mode"],
        generation_mode=config["generation_mode"],
        citation_mode=config["citation_mode"],
        citation_block_policy=config["citation_block_policy"],
    )
    print(live["status"], live["answer"])
    for citation in live.get("citations", []):
        print(citation["article"], citation["pdf_page_start"], citation["quote"])
else:
    print("Génération en direct désactivée ; aucun appel au GPU.")


## 9. Lancer l’application sur ce PC

Depuis la racine du projet, dans deux terminaux PowerShell :

```powershell
.\scripts\start_ollama.ps1
# Dans un second terminal :
.\scripts\start_ui.ps1
```

Ouvrir `http://127.0.0.1:8501`, choisir l’édition, poser la question, puis consulter les citations et les pages PDF. La commande `start_ui.ps1` utilise désormais les quatre éditions. Pour arrêter Ollama après la séance : `.\scripts\start_ollama.ps1 -Stop`.

Les traces des essais sont dans `data/evaluation/` et `data/mlflow/` local`. Une édition du CGI ne suffit pas à elle seule pour déterminer la règle applicable à une opération datée : les dispositions transitoires peuvent être nécessaires.
